导入+配置

In [1]:
import re
import os

INPUT_FILE = "MINI_pattern_match_snort3_content.txt"
OUTPUT_FILE = "/home/m2_1/dat480_project_base/Project_kernels_HLS/src/patterns.h"

if not os.path.exists(INPUT_FILE):
    if os.path.exists("../" + INPUT_FILE):
        INPUT_FILE = "../" + INPUT_FILE
        print(f"在上一级目录找到了文件: {INPUT_FILE}")
    else:
        print(f"错误: 找不到输入文件 {INPUT_FILE}")
else:
    print(f"找到输入文件: {INPUT_FILE}")



找到输入文件: MINI_pattern_match_snort3_content.txt


检查是否是真 HEX
解析每一行的 Snort 规则
    - |0A 0D| → 真Hex
    - |Chrome| → 文本
    - 普通字符串

In [2]:
def is_valid_hex_content(s):
    if not s:
        return False
    valid_chars = set('0123456789abcdefABCDEF ')
    return all(c in valid_chars for c in s)

def parse_snort_rule(line):
    line = line.strip()
    if not line:
        return None

    parts = re.split(r'(\|[^|]+\|)', line) 
    byte_array = []

    for part in parts:
        if part.startswith('|') and part.endswith('|') and len(part) > 2:
            inner = part[1:-1]

            if is_valid_hex_content(inner):
                # 真 HEX
                hex_content = inner.replace(" ", "")
                try:
                    for i in range(0, len(hex_content), 2):
                        byte_array.append(int(hex_content[i:i+2], 16))
                except ValueError:
                    for c in part:
                        byte_array.append(ord(c))
            else:
                # |xxx|
                for c in part:
                    byte_array.append(ord(c))
        else:
            for c in part:
                byte_array.append(ord(c))

    return byte_array if byte_array else None

生成 patterns.h

In [3]:
def generate_cpp_header():
    patterns = []
    print(f"正在读取 {INPUT_FILE} ...")

    try:
        with open(INPUT_FILE, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                if line.startswith(';\n'):
                    continue
                if not line.strip():
                    continue

                result = parse_snort_rule(line)
                if result:
                    patterns.append(result)

        if not patterns:
            print("没有读取到任何模式，已取消生成。")
            return

        max_len = max(len(p) for p in patterns)

        with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
            f.write("#ifndef PATTERNS_H\n#define PATTERNS_H\n\n")
            f.write(f"#define NUM_PATTERNS {len(patterns)}\n")
            f.write(f"#define PATTERN_MAX_LEN {max_len}\n\n")

            f.write("typedef struct {\n")
            f.write("    unsigned char data[PATTERN_MAX_LEN];\n")
            f.write("    int len;\n")
            f.write("} Pattern;\n\n")


            f.write("static const Pattern rules[NUM_PATTERNS] = {\n")
            for p in patterns:
                hex_str = ", ".join([f"0x{b:02X}" for b in p])
                padding = ", ".join(["0x00"] * (max_len - len(p)))
                if padding:
                    hex_str = hex_str + ", " + padding
                f.write(f"    {{ {{ {hex_str} }}, {len(p)} }},\n")
            f.write("};\n\n")
            f.write("#endif // PATTERNS_H\n")

        print(f"成功！已生成 {OUTPUT_FILE} (共 {len(patterns)} 个 pattern)")

    except Exception as e:
        print(f"写入文件失败: {e}")



执行

In [4]:
generate_cpp_header()

正在读取 MINI_pattern_match_snort3_content.txt ...
成功！已生成 /home/m2_1/dat480_project_base/Project_kernels_HLS/src/patterns.h (共 256 个 pattern)
